In [189]:
import os
import re
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

from tqdm import tqdm

import lightgbm as lgb
import shap

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.metrics import roc_curve

# 윈도우 기본 한글 폰트 설정
mpl.rcParams["font.family"] = "Malgun Gothic"  # 맑은 고딕 사용

# 마이너스 깨짐 방지
mpl.rcParams["axes.unicode_minus"] = False  # 음수 기호 정상 출력

In [ ]:
"""
# 1. match_meta
meta_path = r"F:\porti\2025\스파르타자료들\python_study\venv\04_Team8\0.googledrive\match_detail\match_meta\*.parquet"
match_meta_df = pd.concat([pd.read_parquet(f) for f in glob.glob(meta_path)], ignore_index=True)

# 2. participants
participants_path = r"F:\porti\2025\스파르타자료들\python_study\venv\04_Team8\0.googledrive\match_detail\participants\*.parquet"
participants_df = pd.concat([pd.read_parquet(f) for f in glob.glob(participants_path)], ignore_index=True)

# 3. teams
teams_path = r"F:\porti\2025\스파르타자료들\python_study\venv\04_Team8\0.googledrive\match_detail\teams\*.parquet"
teams_df = pd.concat([pd.read_parquet(f) for f in glob.glob(teams_path)], ignore_index=True)

# 4. events
events_path = r"F:\porti\2025\스파르타자료들\python_study\venv\04_Team8\0.googledrive\match_timeline\events\*.parquet"
events_df = pd.concat([pd.read_parquet(f) for f in glob.glob(events_path)], ignore_index=True)

# 5. frames
frames_path = r"F:\porti\2025\스파르타자료들\python_study\venv\04_Team8\0.googledrive\match_timeline\frames\*.parquet"
frames_df = pd.concat([pd.read_parquet(f) for f in glob.glob(frames_path)], ignore_index=True)
"""

In [193]:
df_kor = pd.read_csv(r"F:\porti\2025\스파르타자료들\python_study\venv\04_Team8\0.googledrive\Youtube\youtube_국내.csv")
df_over = pd.read_csv(r"F:\porti\2025\스파르타자료들\python_study\venv\04_Team8\0.googledrive\Youtube\youtube_해외(번역본).csv")
df_reddit = pd.read_csv(r"F:\porti\2025\스파르타자료들\python_study\venv\04_Team8\0.googledrive\Reddit\translated_progress_gcloud_ver.csv")

print("국내 유튜브 컬럼")
print(df_kor.columns)
print("\n해외 유튜브 번역본 컬럼")
print(df_over.columns)
print("\nReddit 번역본 컬럼")
print(df_reddit.columns)

국내 유튜브 컬럼
Index(['playlist_title', 'video_id', 'title', 'description',
       'publish_time_utc', 'viewCount', 'likeCount', 'commentCount',
       'comment_text', 'comment_likeCount'],
      dtype='object')

해외 유튜브 번역본 컬럼
Index(['playlist_title', 'video_id', 'title', 'description', 'Date Posted',
       'viewCount', 'likeCount', 'commentCount', 'comment_text',
       'comment_likeCount', 'has_emoji'],
      dtype='object')

Reddit 번역본 컬럼
Index(['subreddit', 'post_id', 'post_title', 'post_created_dt', 'post_score',
       'post_num_comments', 'comment_id', 'comment_author', 'comment_score',
       'comment_created_dt', 'comment_body_raw', 'comment_body_clean',
       'emoji_list', 'emoji_count', 'language', 'has_emoji',
       'translated_text'],
      dtype='object')


In [318]:
df_kor

,playlist_title,video_id,title,description,publish_time_utc,viewCount,likeCount,commentCount,comment_text,comment_likeCount,comment,date,source
0,롤 YT 분석 2,ywNPhRUYnzk,"시즌 999호 롤망함 스타트 , 녹서스 새 시즌 모든 변경점 총정리",안녕하세요 해도리입니다!\n\n곧 적용될 리그 오브 레전드 25시즌 대격변 변경점들...,2025-01-01T00:30:46Z,472851,2947,1248,8:46 신분제 도입,1046,8:46 신분제 도입,2025-01-01T00:30:46Z,youtube
1,롤 YT 분석 2,ywNPhRUYnzk,"시즌 999호 롤망함 스타트 , 녹서스 새 시즌 모든 변경점 총정리",안녕하세요 해도리입니다!\n\n곧 적용될 리그 오브 레전드 25시즌 대격변 변경점들...,2025-01-01T00:30:46Z,472851,2947,1248,명예 5렙상점 보상이나 추가해줬으면... 명예 5렙토큰만 4개 쌓여 있는데 트위치랑...,598,명예 5렙상점 보상이나 추가해줬으면... 명예 5렙토큰만 4개 쌓여 있는데 트위치랑...,2025-01-01T00:30:46Z,youtube
2,롤 YT 분석 2,ywNPhRUYnzk,"시즌 999호 롤망함 스타트 , 녹서스 새 시즌 모든 변경점 총정리",안녕하세요 해도리입니다!\n\n곧 적용될 리그 오브 레전드 25시즌 대격변 변경점들...,2025-01-01T00:30:46Z,472851,2947,1248,다들 새해 복 많이 받아라,514,다들 새해 복 많이 받아라,2025-01-01T00:30:46Z,youtube
3,롤 YT 분석 2,ywNPhRUYnzk,"시즌 999호 롤망함 스타트 , 녹서스 새 시즌 모든 변경점 총정리",안녕하세요 해도리입니다!\n\n곧 적용될 리그 오브 레전드 25시즌 대격변 변경점들...,2025-01-01T00:30:46Z,472851,2947,1248,근데 신발은 한팀만 강화시키는건 진짜 안좋은 판단같음... 지금처럼 무료로 업그레이...,247,근데 신발은 한팀만 강화시키는건 진짜 안좋은 판단같음... 지금처럼 무료로 업그레이...,2025-01-01T00:30:46Z,youtube
4,롤 YT 분석 2,ywNPhRUYnzk,"시즌 999호 롤망함 스타트 , 녹서스 새 시즌 모든 변경점 총정리",안녕하세요 해도리입니다!\n\n곧 적용될 리그 오브 레전드 25시즌 대격변 변경점들...,2025-01-01T00:30:46Z,472851,2947,1248,나이스 드디어 명예레벨 낮은 것들이 채팅을 못치는 시대가 오는고만,232,나이스 드디어 명예레벨 낮은 것들이 채팅을 못치는 시대가 오는고만,2025-01-01T00:30:46Z,youtube
...,...,...,...,...,...,...,...,...,...,...,...,...,...
24884,LOL YT 분석,lV50lGwQVfo,"26시즌, 솔랭은 흥했을까?",아직도 안 챙기셨어요? 148만원 가만히 두면 손해!💸\n👉 내 지원금 알아보기 h...,2026-02-18T14:31:33Z,62289,552,466,3,0,3,2026-02-18T14:31:33Z,youtube
24885,LOL YT 분석,lV50lGwQVfo,"26시즌, 솔랭은 흥했을까?",아직도 안 챙기셨어요? 148만원 가만히 두면 손해!💸\n👉 내 지원금 알아보기 h...,2026-02-18T14:31:33Z,62289,552,466,솔직히 저번시즌보다 없어짐 제재가 먹히긴 하나봄ㅋㅋ,0,솔직히 저번시즌보다 없어짐 제재가 먹히긴 하나봄ㅋㅋ,2026-02-18T14:31:33Z,youtube
24886,LOL YT 분석,lV50lGwQVfo,"26시즌, 솔랭은 흥했을까?",아직도 안 챙기셨어요? 148만원 가만히 두면 손해!💸\n👉 내 지원금 알아보기 h...,2026-02-18T14:31:33Z,62289,552,466,"칼바람 아수라장 하다가 랭크도 부가적으로 하게됨 , \n억까매칭은 여전히 있는게 아...",0,"칼바람 아수라장 하다가 랭크도 부가적으로 하게됨 , \n억까매칭은 여전히 있는게 아...",2026-02-18T14:31:33Z,youtube
24887,LOL YT 분석,lV50lGwQVfo,"26시즌, 솔랭은 흥했을까?",아직도 안 챙기셨어요? 148만원 가만히 두면 손해!💸\n👉 내 지원금 알아보기 h...,2026-02-18T14:31:33Z,62289,552,466,2,0,2,2026-02-18T14:31:33Z,youtube


In [194]:
# YouTube 국내 / 해외 구분
df_kor["region_type"] = "국내"
df_over["region_type"] = "해외"

# YouTube 통합
df_kor["comment"] = df_kor["comment_text"]
df_over["comment"] = df_over["comment_text"]

df_kor["date"] = df_kor["publish_time_utc"]
df_over["date"] = df_over["Date Posted"]

df_kor["source"] = "youtube"
df_over["source"] = "youtube"

df_youtube = pd.concat([df_kor, df_over], ignore_index=True)

# Reddit → 해외 고정
df_reddit["comment"] = df_reddit["translated_text"]
df_reddit["date"] = df_reddit["comment_created_dt"]
df_reddit["source"] = "reddit"
df_reddit["region_type"] = "해외"

# 전체 통합
df_all = pd.concat([df_youtube, df_reddit], ignore_index=True)

# 날짜 전처리
def clean_date(value):
    if pd.isna(value):
        return None
    
    try:
        dt = pd.to_datetime(value, errors="coerce")
        
        if pd.isna(dt):
            return None
        
        # timezone 제거 (시간 유지)
        if hasattr(dt, "tzinfo") and dt.tzinfo is not None:
            dt = dt.tz_convert(None)
        
        # normalize() 제거 시간 유지
        return dt.strftime("%Y-%m-%d %H:%M:%S")
    
    except:
        return None
        
df_all["date"] = df_all["date"].apply(clean_date)

# ADC -> 원거리 딜러 통일
def normalize_adc(text):
    if pd.isna(text):
        return text
    
    text = str(text)
    text = re.sub(r"\(ADC\)", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\bADC\b", "원거리 딜러", text, flags=re.IGNORECASE)
    return text.strip()

df_all["comment"] = df_all["comment"].apply(normalize_adc)

In [197]:
df_all

,playlist_title,video_id,title,description,publish_time_utc,viewCount,likeCount,commentCount,comment_text,comment_likeCount,...,comment_id,comment_author,comment_score,comment_created_dt,comment_body_raw,comment_body_clean,emoji_list,emoji_count,language,translated_text
0,롤 YT 분석 2,ywNPhRUYnzk,"시즌 999호 롤망함 스타트 , 녹서스 새 시즌 모든 변경점 총정리",안녕하세요 해도리입니다!\n\n곧 적용될 리그 오브 레전드 25시즌 대격변 변경점들...,2025-01-01T00:30:46Z,472851.0,2947.0,1248.0,8:46 신분제 도입,1046.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,롤 YT 분석 2,ywNPhRUYnzk,"시즌 999호 롤망함 스타트 , 녹서스 새 시즌 모든 변경점 총정리",안녕하세요 해도리입니다!\n\n곧 적용될 리그 오브 레전드 25시즌 대격변 변경점들...,2025-01-01T00:30:46Z,472851.0,2947.0,1248.0,명예 5렙상점 보상이나 추가해줬으면... 명예 5렙토큰만 4개 쌓여 있는데 트위치랑...,598.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,롤 YT 분석 2,ywNPhRUYnzk,"시즌 999호 롤망함 스타트 , 녹서스 새 시즌 모든 변경점 총정리",안녕하세요 해도리입니다!\n\n곧 적용될 리그 오브 레전드 25시즌 대격변 변경점들...,2025-01-01T00:30:46Z,472851.0,2947.0,1248.0,다들 새해 복 많이 받아라,514.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,롤 YT 분석 2,ywNPhRUYnzk,"시즌 999호 롤망함 스타트 , 녹서스 새 시즌 모든 변경점 총정리",안녕하세요 해도리입니다!\n\n곧 적용될 리그 오브 레전드 25시즌 대격변 변경점들...,2025-01-01T00:30:46Z,472851.0,2947.0,1248.0,근데 신발은 한팀만 강화시키는건 진짜 안좋은 판단같음... 지금처럼 무료로 업그레이...,247.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,롤 YT 분석 2,ywNPhRUYnzk,"시즌 999호 롤망함 스타트 , 녹서스 새 시즌 모든 변경점 총정리",안녕하세요 해도리입니다!\n\n곧 적용될 리그 오브 레전드 25시즌 대격변 변경점들...,2025-01-01T00:30:46Z,472851.0,2947.0,1248.0,나이스 드디어 명예레벨 낮은 것들이 채팅을 못치는 시대가 오는고만,232.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
152945,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,nlfvy8a,The_God_of_Biscuits,0.0,2025-10-26 07:18:25,"If you have prio, you can crash whenever if jg...","if you have prio, you can crash whenever if jg...",[],0.0,en,"우선권이 있다면, 정글러가 주변에 없을 때 언제든 공격을 시작할 수 있고, 그 후 ..."
152946,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,nrv6swj,Desperate-Zebra-3855,15.0,2025-12-02 10:51:24,"As an example, Aatrox gets roughly 5 ad per le...","as an example, aatrox gets roughly 5 ad per le...",[],0.0,en,"예를 들어, 아트록스는 레벨당 대략 5의 공격력을 얻습니다. 레벨이 2만큼 더 오르..."
152947,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,nrw4lvq,Active-Advisor5909,3.0,2025-12-02 14:45:32,Steraks will be slightly better. But a bonus 6...,steraks will be slightly better. but a bonus 6...,[],0.0,en,스테락스가 약간 더 나을 겁니다. 하지만 보너스 6 광고가 있다고 해서 대부분의 아...
152948,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,nsg25jd,Mindless-Humor-9036,1.0,2025-12-05 16:21:43,I think you’re over thinking it tbh. I only go...,i think you’re over thinking it tbh. i only go...,[],0.0,en,"솔직히 너무 과하게 생각하시는 것 같아요. 전 인내력이 필요할 때만 연승을 가고, ..."


In [ ]:
# 분류 키워드 정의
champion_list = [
    '가렌', '갈리오', '갱플랭크', '갱플', '그라가스', '그레이브즈', '그브', '그웬', 
    '나르', '나미', '나서스', '나피리', '노틸러스', '노틸', '녹턴', '누누와 윌럼프', '누누', '니달리', '니코', '닐라', 
    '다리우스', '다이애나', '드레이븐', '드븐',
    '라이즈', '라칸', '람머스', '럭스', '럼블', '레나타 글라스크', '레나타', '레넥톤', '레오나', '렉사이', '렐', 
    '렝가', '루시안', '룰루', '르블랑', '리 신', '리신', '리븐', '리산드라', '리산', '릴리아', 
    '마스터 이','마스터이', '마이', '마오카이', '말자하', '말파이트', '멜', '모데카이저', '모데','모르가나', '몰가', 
    '문도 박사', '문도', '문도박사', '미스 포츈', '미스포춘', '미포', '밀리오', 
    '바드', '바루스', '바이', '베이가', '베인', '벡스', '백스', '벨베스', '벨코즈', 
    '볼리베어', '볼베', '브라움', '브라이어', '브랜드', '블라디미르', '블라디', '블라드','블리츠크랭크', '블리츠', '블츠', '비에고', '빅토르', '뽀삐', 
    '사미라', '사이온', '사일러스', '사일', '샤코', '세나', '세라핀', '세주아니', '세주', '세트', 
    '소나', '소라카', '쉔', '쉬바나', '스몰더', '스웨인', '스카너', '시비르', '신 짜오', '신짜오', '짜오', '신드라', '신지드', '쓰레쉬', 
    '아리', '아무무', '아우렐리온 솔', '아솔', '아우솔', '아이번', '아지르', '아칼리', '아크샨', 
    '아트록스', '아펠리오스', '아펠', '알리스타', '알리', '암베사', '애니', '애니비아', '애쉬', 
    '야스오', '야소', '에코', '엘리스', '오공', '손오공', '오로라', '오른', '오리아나', '올라프', 
    '요네', '요릭', '우디르', '우르곳', '워윅', '유나라', '유미', '이렐리아', '이렐', '이블린', '이즈리얼', '이즈', '일라오이', '일라',
    '자르반 4세', '자르반4세', '자르반', '자야', '자이라', '자크', '자헨', '잔나', '잭스', '제드', 
    '제라스', '제리', '제이스', '조이', '직스', '진', '질리언', '징크스', '초가스', 
    '카르마', '카밀', '카사딘', '카서스', '카시오페아', '카시', '카이사', '카직스', '카타리나', '카타',
    '칼리스타', '칼리', '케넨', '케이틀린', '케틀', '케인', '케일', '코그모', '코르키', '퀸', '크산테', '클레드', '키아나', '킨드레드', '킨드',
    '타릭', '탈론', '탈리야', '탐 켄치', '탐켄치', '켄치', '트런들', '트리스타나', '트타', '트린다미어', '트린', '트위스티드 페이트', '트페', '트위치', '티모', 
    '파이크', '판테온', '판테', '피들스틱', '피들', '피오라', '피즈', '하이머딩거', '하이머', '하딩', '딩거', '헤카림', '흐웨이'
]

position_dict = {
    "탑": ["탑"],
    "정글": ["정글", "정글러"],
    "미드": ["미드"],
    "원거리 딜러": ["원거리 딜러", "원딜", "바텀"],
    "서포터": ["서포터", "서폿"]
}

cause_keywords = [
    "부담", "압박", "책임",
    "격차", "차이", "밀리",
    "시야", "와드", "맵",
    "자동 배정", "오토필"
]

performance_keywords = [
    "딜", "데미지", "kda",
    "시야 점수", "오브젝트",
    "골드", "cs", "아이템", "갱"
]

stress_keywords = [
    "스트레스", "짜증", "화남",
    "피곤", "지침", "지쳐",
    "답답", "증오","개노잼"
]

positive_keywords = [
    "좋", "잘", "강",
    "사기", "캐리", "쎄",
    "재밌", "괜찮", "최고",
    "든든", "완벽", "쉽다",
    "편안", "공감", "강력",
    "안전", "추천", "자랑",
    "유용", "적합", "효과적",
    "장인", "온전", "성공",
    "건재", "이쁘네", "예뻐요",
    "즐거워요","기쁩니다", "설렌다",
    "맞다","개꿀이네","멋있는데",
    "귀엽다","편해","개꿀",
    "만족스러워요"
]

negative_keywords = [
    "망","망겜", "별로", "문제",
    "불공평", "어렵", "힘들",
    "어려", "사고", "짜증",
    "싫어","싫었","도무지",
    "답답","빌어먹을","없던",
    "안돼","안 돼","없애",
    "트롤","모욕","더러운",
    "좁아","없네","최악",
    "않은", "못","없는",
    "끔찍", "착각",
    "부족","없","슬픈",
    "박살", "쓰레기", "까다롭",
    "카이팅","욕","무시",
    "증오","슬퍼","안타깝",
    "망상","종료", "떨어진",
    "접어","오바","엠창",
    "저딴","초딩","억지로",
    "난리","조작","않음",
    "사형","힘든듯","노잼",
    "강제","운빨","에바",
    "경악", "꼴보기싫다", "역겹다",
    "기괴", "엿먹어", "구려",
    "참담한", "멍청한","느려진",
    "구리네","필요없","적폐",
    "토나오네","약함", "짜쳐",
    "짜친다", "짜치", "안됨",
    "떠남", "중국겜","뻔뻔",
    "역하네", "불쾌", "오열",
    "실직"
]

category_keywords = {
    "포지션 밸런스": [
        "탑", "정글", "미드", "봇", "봇전", "봇라인", "바텀", "원거리 딜러", "원딜", "서폿", "라인전", "라인 차이", "라인"
    ],
    
    "챔피언 밸런스": [
        "사기", "너프", "버프", "OP", "고인", "밸런스", "챔프", "후반챔", "신챔", "신챔프", "패시브", "AP", "AD", "날먹챔",
        "주챔프", "주챔", "벨런스", "ap", "ad", "op"
    ],

    "게임 시스템": [
        "오토필", "자동 배정", "패치", "매칭", "랭크", "MMR", "드래곤", "유충", "시즌", "배치", "인게임", "리메이크",
        "시스템", "큐", "업데이트", "칼바람", "디도스", "버그", "그래픽", "명예", "MVP","WASD", "wasd", "티어", "랭겜", 
        "어뷰징", "뱅가드", "벵가드", "인겜", "솔랭", "자랭", "클라렉"
    ],
    "전투/데미지": [
        "딜", "데미지", "원콤", "카이팅", "아이템", "전투", "점화", "쿼드라", "쿼드라킬", "평타",
        "골드", "CS", "오브젝트", "갱", "한타", "미니언", "펜타킬", "주문력", "궁극기", "파밍",
        "공속", "광폭화", "장로", "전령", "바론", "상위템", "스킬", "실직"
    ],
    "팀원/유저 행동": [
        "트롤", "욕", "정치", "팀운", "모욕", "탈주", "싸움", "쌈", "라인스왑", "다이브", "서렌", "닷지", "던지기"
    ],
    "게임 전반 감정": [
        "스트레스", "짜증", "화남", "피곤","ㅜㅜ","개노잼","에바",
        "지침", "망겜", "재밌", "최악","엠창","ㅈㄴ","노잼","즐거워요",
        "경악", "꼴보기싫다","역겹다","기괴", "구려","참담한","기쁩니다","구리네",
        "설렌다", "개꿀이네","토나오네","멋있는데","귀엽다","중국겜","개꿀","만족",
        "역하네", "불쾌", "오열"
    ]
}

In [236]:
df_all["comment"] = df_all["comment"].str.replace(
    r"k['’]?sante", 
    "크산테", 
    case=False, 
    regex=True
)

df_all["comment"] = df_all["comment"].str.replace(
    "모르데카이저",
    "모데카이저",
    regex=False
)

df_all["comment"] = df_all["comment"].str.replace(
    "모르데카이",
    "모데카이저",
    regex=False
)

df_all["comment"] = df_all["comment"].str.replace(
    "시온",
    "사이온",
    regex=False
)

df_all["comment"] = df_all["comment"].str.replace(
    "sion",
    "사이온",
    regex=False
)

df_all["comment"] = df_all["comment"].str.replace(
    "Jax",
    "잭스",
    regex=False
)

df_all["comment"] = df_all["comment"].str.replace(
    "GP",
    "갱플랭크",
    regex=False
)

df_all["comment"] = df_all["comment"].str.replace(
    "gp",
    "갱플랭크",
    regex=False
)

df_all["comment"] = df_all["comment"].str.replace(
    "gragas",
    "그라가스",
    regex=False
)

df_all["comment"] = df_all["comment"].str.replace(
    "포피",
    "뽀삐",
    regex=False
)

df_all["comment"] = df_all["comment"].str.replace(
    "싱드",
    "신지드",
    regex=False
)

df_all["comment"] = df_all["comment"].str.replace(
    "신지",
    "신지드",
    regex=False
)

df_all["comment"] = df_all["comment"].str.replace(
    "나수스",
    "나서스",
    regex=False
)

df_all["comment"] = df_all["comment"].str.replace(
    "신자오",
    "신짜오",
    regex=False
)

df_all["comment"] = df_all["comment"].str.replace(
    "신 자오",
    "신짜오",
    regex=False
)

df_all["comment"] = df_all["comment"].str.replace(
    "오르엔",
    "오른",
    regex=False
)

df_all["comment"] = df_all["comment"].str.replace(
    "말자하르",
    "말자하",
    regex=False
)

df_all["comment"] = df_all["comment"].str.replace(
    "칼리스트",
    "칼리스타",
    regex=False
)

df_all["comment"] = df_all["comment"].str.replace(
    "그나르",
    "나르",
    regex=False
)

df_all["comment"] = df_all["comment"].str.replace(
    "mundo with hs, mog, visage, fon, dmp는 잘 작동할 것입니다.",
    "문도가 hs, mog, visage, fon, dmp를 갔을 때는 잘 작동할 것입니다.",
    regex=False
)


In [237]:
# 주 포지션 추출
def extract_main_position(text):
    if pd.isna(text):
        return "기타"

    text = str(text)
    counts = {}

    for pos, keywords in position_dict.items():
        count = 0
        for word in keywords:
            count += len(re.findall(word, text))
        if count > 0:
            counts[pos] = count

    if not counts:
        return "기타"

    return max(counts, key=counts.get)

df_all["main_position"] = df_all["comment"].apply(extract_main_position)

In [238]:
# 조사 패턴 정의
josa_pattern = "(은|는|이|가|을|를|와|과|도|만|에|에서|에게|에게는|한테|한테서|로|로도|으로|의|나|라고|라면|부터|까지|야|이라는|이나|처럼|만큼|랑|고|보다|좀|님|충|행)?"

# 카테고리 분류 함수
def classify_comment(text):
    if pd.isna(text):
        return "기타"

    text = str(text)

    # 챔피언 언급 감지
    mentioned = False
    for champ in champion_list:
        pattern = rf"(?<![가-힣]){re.escape(champ)}{josa_pattern}(?![가-힣])"
        if re.search(pattern, text):
            mentioned = True
            break

    # 챔피언 밸런스 키워드 사용
    balance_words = category_keywords["챔피언 밸런스"]
    has_balance_word = any(word.lower() in text.lower() for word in balance_words)

    # 우선순위 처리
    if mentioned and has_balance_word:
        return "챔피언 밸런스"

    if mentioned:
        return "챔피언 언급"

    # 나머지 기존 카테고리 처리
    for category, keywords in category_keywords.items():
        if category == "챔피언 밸런스":
            continue

        for word in keywords:
            if word in text:
                return category

    return "기타"

df_all["category"] = df_all["comment"].apply(classify_comment)

In [239]:
# 불만지수 계산
def calculate_complaint_index(text):
    if pd.isna(text):
        return 0

    text = str(text)
    score = 0

    for word in positive_keywords:
        if word in text:
            score += 1

    for word in negative_keywords:
        if word in text:
            score -= 1

    for word in stress_keywords:
        if word in text:
            score -= 2

    return score

df_all["complaint_index"] = df_all["comment"].apply(calculate_complaint_index)

In [240]:
# 공통 7개 컬럼만 남기기
df_final = df_all[["date", "source", "region_type", "comment", "main_position", "category", "complaint_index"]]

df_final

,date,source,region_type,comment,main_position,category,complaint_index
0,2025-01-01 00:30:46,youtube,국내,8:46 신분제 도입,기타,기타,0
1,2025-01-01 00:30:46,youtube,국내,명예 5렙상점 보상이나 추가해줬으면... 명예 5렙토큰만 4개 쌓여 있는데 트위치랑...,기타,챔피언 언급,-1
2,2025-01-01 00:30:46,youtube,국내,다들 새해 복 많이 받아라,기타,기타,0
3,2025-01-01 00:30:46,youtube,국내,근데 신발은 한팀만 강화시키는건 진짜 안좋은 판단같음... 지금처럼 무료로 업그레이...,기타,전투/데미지,1
4,2025-01-01 00:30:46,youtube,국내,나이스 드디어 명예레벨 낮은 것들이 채팅을 못치는 시대가 오는고만,기타,게임 시스템,-1
...,...,...,...,...,...,...,...
152945,2025-10-26 07:18:25,reddit,해외,"우선권이 있다면, 정글러가 주변에 없을 때 언제든 공격을 시작할 수 있고, 그 후 ...",정글,포지션 밸런스,-1
152946,2025-12-02 10:51:24,reddit,해외,"예를 들어, 아트록스는 레벨당 대략 5의 공격력을 얻습니다. 레벨이 2만큼 더 오르...",기타,챔피언 언급,-1
152947,2025-12-02 14:45:32,reddit,해외,스테락스가 약간 더 나을 겁니다. 하지만 보너스 6 광고가 있다고 해서 대부분의 아...,기타,전투/데미지,0
152948,2025-12-05 16:21:43,reddit,해외,"솔직히 너무 과하게 생각하시는 것 같아요. 전 인내력이 필요할 때만 연승을 가고, ...",기타,기타,0


In [241]:
# html 코드 기호 전처리
import html
df_final["comment"] = df_final["comment"].apply(lambda x: html.unescape(x) if pd.notna(x) else x)
df_final

C:\Users\user\AppData\Local\Temp/ipykernel_32796/22810719.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_final["comment"] = df_final["comment"].apply(lambda x: html.unescape(x) if pd.notna(x) else x)


,date,source,region_type,comment,main_position,category,complaint_index
0,2025-01-01 00:30:46,youtube,국내,8:46 신분제 도입,기타,기타,0
1,2025-01-01 00:30:46,youtube,국내,명예 5렙상점 보상이나 추가해줬으면... 명예 5렙토큰만 4개 쌓여 있는데 트위치랑...,기타,챔피언 언급,-1
2,2025-01-01 00:30:46,youtube,국내,다들 새해 복 많이 받아라,기타,기타,0
3,2025-01-01 00:30:46,youtube,국내,근데 신발은 한팀만 강화시키는건 진짜 안좋은 판단같음... 지금처럼 무료로 업그레이...,기타,전투/데미지,1
4,2025-01-01 00:30:46,youtube,국내,나이스 드디어 명예레벨 낮은 것들이 채팅을 못치는 시대가 오는고만,기타,게임 시스템,-1
...,...,...,...,...,...,...,...
152945,2025-10-26 07:18:25,reddit,해외,"우선권이 있다면, 정글러가 주변에 없을 때 언제든 공격을 시작할 수 있고, 그 후 ...",정글,포지션 밸런스,-1
152946,2025-12-02 10:51:24,reddit,해외,"예를 들어, 아트록스는 레벨당 대략 5의 공격력을 얻습니다. 레벨이 2만큼 더 오르...",기타,챔피언 언급,-1
152947,2025-12-02 14:45:32,reddit,해외,스테락스가 약간 더 나을 겁니다. 하지만 보너스 6 광고가 있다고 해서 대부분의 아...,기타,전투/데미지,0
152948,2025-12-05 16:21:43,reddit,해외,"솔직히 너무 과하게 생각하시는 것 같아요. 전 인내력이 필요할 때만 연승을 가고, ...",기타,기타,0


In [242]:
# 레딧에서 삭제됨 포함된 댓글 필터링
url_mask = df_final["comment"].str.contains("레딧에서 삭제됨", na=False)

# 삭제 대상 따로 저장
deleted_comments = df_final[url_mask]

print("===== 삭제될 댓글 목록 =====")
print(f"총 삭제 개수: {len(deleted_comments)}\n")

for idx, row in deleted_comments.iterrows():
    print(f"[인덱스: {idx}]")
    print(f"날짜: {row.get('date', 'N/A')}")
    print(f"카테고리: {row.get('category', 'N/A')}")
    print(f"댓글 내용: {row['comment']}")
    print("-" * 80)

===== 삭제될 댓글 목록 =====
총 삭제 개수: 19

[인덱스: 42254]
날짜: 2025-08-16 07:48:45
카테고리: 기타
댓글 내용: [레딧에서 삭제됨]
--------------------------------------------------------------------------------
[인덱스: 45992]
날짜: 2025-06-25 14:18:13
카테고리: 기타
댓글 내용: [레딧에서 삭제됨]
--------------------------------------------------------------------------------
[인덱스: 47452]
날짜: 2026-01-29 13:10:10
카테고리: 기타
댓글 내용: [레딧에서 삭제됨]
--------------------------------------------------------------------------------
[인덱스: 48999]
날짜: 2025-09-06 04:52:38
카테고리: 기타
댓글 내용: [레딧에서 삭제됨]
--------------------------------------------------------------------------------
[인덱스: 51215]
날짜: 2025-12-11 14:28:31
카테고리: 기타
댓글 내용: [레딧에서 삭제됨]
--------------------------------------------------------------------------------
[인덱스: 51479]
날짜: 2025-10-24 09:56:01
카테고리: 기타
댓글 내용: [레딧에서 삭제됨]
--------------------------------------------------------------------------------
[인덱스: 54577]
날짜: 2025-12-02 04:17:45
카테고리: 기타
댓글 내용: [레딧에서 삭제됨]
-------------------------------

In [243]:
# https:// 포함된 댓글 필터링
url_mask2 = df_final["comment"].str.contains("https://", na=False)

# 삭제 대상 따로 저장
deleted_comments = df_final[url_mask2]

print("===== 삭제될 댓글 목록 =====")
print(f"총 삭제 개수: {len(deleted_comments)}\n")

for idx, row in deleted_comments.iterrows():
    print(f"[인덱스: {idx}]")
    print(f"날짜: {row.get('date', 'N/A')}")
    print(f"카테고리: {row.get('category', 'N/A')}")
    print(f"댓글 내용: {row['comment']}")
    print("-" * 80)

===== 삭제될 댓글 목록 =====
총 삭제 개수: 15

[인덱스: 517]
날짜: 2025-01-21 05:33:25
카테고리: 기타
댓글 내용: https://nordvpn.com/haedory
2025 신년 특가 On💡
74% 할인 + 4개월 무료 추가 (쿠폰코드: haedory)
10대 동시 연결! 한국 고정 IP 단독 제공! 원클릭 보안!
--------------------------------------------------------------------------------
[인덱스: 519]
날짜: 2025-01-21 05:33:25
카테고리: 기타
댓글 내용: https://nordvpn.com/haedory
2025 신년 특가 On💡
74% 할인 + 4개월 무료 추가 (쿠폰코드: haedory)
10대 동시 연결! 한국 고정 IP 단독 제공! 원클릭 보안!
--------------------------------------------------------------------------------
[인덱스: 3506]
날짜: 2025-03-18 20:48:03
카테고리: 게임 시스템
댓글 내용: 오늘 본섭에 적용된 15.6 패치 https://youtu.be/P_V4YcdaMow
--------------------------------------------------------------------------------
[인덱스: 3509]
날짜: 2025-03-18 20:48:03
카테고리: 게임 시스템
댓글 내용: 오늘 본섭에 적용된 15.6 패치 https://youtu.be/P_V4YcdaMow
--------------------------------------------------------------------------------
[인덱스: 8229]
날짜: 2025-08-12 23:59:13
카테고리: 게임 시스템
댓글 내용: 오늘 본섭에 적용된 25.16 패치
https://youtu.be/nGTqBeGGD1w?

In [244]:
df_final_clean = df_final[~url_mask].reset_index(drop=True)
df_final_clean

,date,source,region_type,comment,main_position,category,complaint_index
0,2025-01-01 00:30:46,youtube,국내,8:46 신분제 도입,기타,기타,0
1,2025-01-01 00:30:46,youtube,국내,명예 5렙상점 보상이나 추가해줬으면... 명예 5렙토큰만 4개 쌓여 있는데 트위치랑...,기타,챔피언 언급,-1
2,2025-01-01 00:30:46,youtube,국내,다들 새해 복 많이 받아라,기타,기타,0
3,2025-01-01 00:30:46,youtube,국내,근데 신발은 한팀만 강화시키는건 진짜 안좋은 판단같음... 지금처럼 무료로 업그레이...,기타,전투/데미지,1
4,2025-01-01 00:30:46,youtube,국내,나이스 드디어 명예레벨 낮은 것들이 채팅을 못치는 시대가 오는고만,기타,게임 시스템,-1
...,...,...,...,...,...,...,...
152926,2025-10-26 07:18:25,reddit,해외,"우선권이 있다면, 정글러가 주변에 없을 때 언제든 공격을 시작할 수 있고, 그 후 ...",정글,포지션 밸런스,-1
152927,2025-12-02 10:51:24,reddit,해외,"예를 들어, 아트록스는 레벨당 대략 5의 공격력을 얻습니다. 레벨이 2만큼 더 오르...",기타,챔피언 언급,-1
152928,2025-12-02 14:45:32,reddit,해외,스테락스가 약간 더 나을 겁니다. 하지만 보너스 6 광고가 있다고 해서 대부분의 아...,기타,전투/데미지,0
152929,2025-12-05 16:21:43,reddit,해외,"솔직히 너무 과하게 생각하시는 것 같아요. 전 인내력이 필요할 때만 연승을 가고, ...",기타,기타,0


In [245]:
df_final_clean_all = df_final_clean[~url_mask2].reset_index(drop=True)
df_final_clean_all

C:\Users\user\AppData\Local\Temp/ipykernel_32796/454285800.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_final_clean_all = df_final_clean[~url_mask2].reset_index(drop=True)


,date,source,region_type,comment,main_position,category,complaint_index
0,2025-01-01 00:30:46,youtube,국내,8:46 신분제 도입,기타,기타,0
1,2025-01-01 00:30:46,youtube,국내,명예 5렙상점 보상이나 추가해줬으면... 명예 5렙토큰만 4개 쌓여 있는데 트위치랑...,기타,챔피언 언급,-1
2,2025-01-01 00:30:46,youtube,국내,다들 새해 복 많이 받아라,기타,기타,0
3,2025-01-01 00:30:46,youtube,국내,근데 신발은 한팀만 강화시키는건 진짜 안좋은 판단같음... 지금처럼 무료로 업그레이...,기타,전투/데미지,1
4,2025-01-01 00:30:46,youtube,국내,나이스 드디어 명예레벨 낮은 것들이 채팅을 못치는 시대가 오는고만,기타,게임 시스템,-1
...,...,...,...,...,...,...,...
152911,2025-10-26 07:18:25,reddit,해외,"우선권이 있다면, 정글러가 주변에 없을 때 언제든 공격을 시작할 수 있고, 그 후 ...",정글,포지션 밸런스,-1
152912,2025-12-02 10:51:24,reddit,해외,"예를 들어, 아트록스는 레벨당 대략 5의 공격력을 얻습니다. 레벨이 2만큼 더 오르...",기타,챔피언 언급,-1
152913,2025-12-02 14:45:32,reddit,해외,스테락스가 약간 더 나을 겁니다. 하지만 보너스 6 광고가 있다고 해서 대부분의 아...,기타,전투/데미지,0
152914,2025-12-05 16:21:43,reddit,해외,"솔직히 너무 과하게 생각하시는 것 같아요. 전 인내력이 필요할 때만 연승을 가고, ...",기타,기타,0


In [246]:
df_final_clean_all.to_csv(
    r"F:\porti\2025\스파르타자료들\python_study\venv\04_Team8\0.googledrive\comment_youtube_reddit_cleaned_data.csv",
    index=False,
    encoding="utf-8-sig"
)

In [247]:
df_read_final_clean = pd.read_csv(r"F:\porti\2025\스파르타자료들\python_study\venv\04_Team8\0.googledrive\comment_youtube_reddit_cleaned_data.csv")
df_read_final_clean

,date,source,region_type,comment,main_position,category,complaint_index
0,2025-01-01 00:30:46,youtube,국내,8:46 신분제 도입,기타,기타,0
1,2025-01-01 00:30:46,youtube,국내,명예 5렙상점 보상이나 추가해줬으면... 명예 5렙토큰만 4개 쌓여 있는데 트위치랑...,기타,챔피언 언급,-1
2,2025-01-01 00:30:46,youtube,국내,다들 새해 복 많이 받아라,기타,기타,0
3,2025-01-01 00:30:46,youtube,국내,근데 신발은 한팀만 강화시키는건 진짜 안좋은 판단같음... 지금처럼 무료로 업그레이...,기타,전투/데미지,1
4,2025-01-01 00:30:46,youtube,국내,나이스 드디어 명예레벨 낮은 것들이 채팅을 못치는 시대가 오는고만,기타,게임 시스템,-1
...,...,...,...,...,...,...,...
152911,2025-10-26 07:18:25,reddit,해외,"우선권이 있다면, 정글러가 주변에 없을 때 언제든 공격을 시작할 수 있고, 그 후 ...",정글,포지션 밸런스,-1
152912,2025-12-02 10:51:24,reddit,해외,"예를 들어, 아트록스는 레벨당 대략 5의 공격력을 얻습니다. 레벨이 2만큼 더 오르...",기타,챔피언 언급,-1
152913,2025-12-02 14:45:32,reddit,해외,스테락스가 약간 더 나을 겁니다. 하지만 보너스 6 광고가 있다고 해서 대부분의 아...,기타,전투/데미지,0
152914,2025-12-05 16:21:43,reddit,해외,"솔직히 너무 과하게 생각하시는 것 같아요. 전 인내력이 필요할 때만 연승을 가고, ...",기타,기타,0


In [248]:
def normalize_singed(text):
    return re.sub(r"(신지드)드+", r"\1", str(text))

In [249]:
df_read_final_clean["comment"] = df_read_final_clean["comment"].apply(normalize_singed)
df_read_final_clean

,date,source,region_type,comment,main_position,category,complaint_index
0,2025-01-01 00:30:46,youtube,국내,8:46 신분제 도입,기타,기타,0
1,2025-01-01 00:30:46,youtube,국내,명예 5렙상점 보상이나 추가해줬으면... 명예 5렙토큰만 4개 쌓여 있는데 트위치랑...,기타,챔피언 언급,-1
2,2025-01-01 00:30:46,youtube,국내,다들 새해 복 많이 받아라,기타,기타,0
3,2025-01-01 00:30:46,youtube,국내,근데 신발은 한팀만 강화시키는건 진짜 안좋은 판단같음... 지금처럼 무료로 업그레이...,기타,전투/데미지,1
4,2025-01-01 00:30:46,youtube,국내,나이스 드디어 명예레벨 낮은 것들이 채팅을 못치는 시대가 오는고만,기타,게임 시스템,-1
...,...,...,...,...,...,...,...
152911,2025-10-26 07:18:25,reddit,해외,"우선권이 있다면, 정글러가 주변에 없을 때 언제든 공격을 시작할 수 있고, 그 후 ...",정글,포지션 밸런스,-1
152912,2025-12-02 10:51:24,reddit,해외,"예를 들어, 아트록스는 레벨당 대략 5의 공격력을 얻습니다. 레벨이 2만큼 더 오르...",기타,챔피언 언급,-1
152913,2025-12-02 14:45:32,reddit,해외,스테락스가 약간 더 나을 겁니다. 하지만 보너스 6 광고가 있다고 해서 대부분의 아...,기타,전투/데미지,0
152914,2025-12-05 16:21:43,reddit,해외,"솔직히 너무 과하게 생각하시는 것 같아요. 전 인내력이 필요할 때만 연승을 가고, ...",기타,기타,0


In [250]:
badword_patterns = [

    # ㅅㅂ 계열 (ㅅ1ㅂ, ㅅ..ㅂ 등)
    r"ㅅ[\W_]*[1lI!|]*[\W_]*ㅂ",
    r"ㅅ[\W_]*[1lI!|]*[\W_]*8",
    r"시[\W_]*[1lI!|]*[\W_]*B*ㅏ",
    r"스[\W_]*[1lI!|]*[\W_]*벌",
    r"슈[\W_]*[1lI!|]*[\W_]*발",

    # 시발 / 씨발 우회 표현 (시1바, 시11바, 씨111바 등)
    r"[시씨][\W_]*[1lI!|]*[\W_]*바",
    r"[시씨][\W_]*[1lI!|]*[\W_]*발",
    r"[시씨][\W_]*[1lI!|]*[\W_]*b",
    r"[십][\W_]*[1lI!|]*[\W_]*알",
    r"[시씨][\W_]*[1lI!|]*[\W_]*팔",
    r"[시씨][\W_]*[1lI!|]*[\W_]*빨",
    r"[시씨][\W_]*[1lI!|]*[\W_]*볼",

    # ㅆㅂ
    r"ㅆ[\W_]*ㅂ",

    # ㅈㄹ 계열
    r"ㅈ[\W_]*[1lI!|]*[\W_]*ㄹ",
    r"[지][\W_]*[1lI!|]*[\W_]*랄",
    r"ㅈ[\W_]*[1lI!|]*[\W_]*랄",
    r"야[\W_]*[1lI!|]*[\W_]*랄",
    r"Z[\W_]*[1lI!|]*[\W_]*랄",

    # 엿 / 엿먹어 계열
    r"엿[\W_]*(먹|먹어)?",

    # ㅂㅅ계열
    r"병신",
    r"ㅂ[\W_]*ㅅ",
    r"병[\W_]*sin",
    r"ㅂ[\W_]*신",
    r"비[\W_]*신",
    r"병[\W_]*ㅅ",
    r"ㅂ[\W_]*싀",
    r"비[\W_]*[1lI!|]*[\W_]*신",

    # ㅈ 직접 표현 비속어
    r"졷",
    r"좃",
    r"좆",

    # 부모님 비속어
    r"[에애][\W_]*미(?=\W|$)",
    r"ㅇ[\W_]*ㅁ[\W_]*ㄷ*ㅈ",

    # 기타 비속어
    r"짜쳐",
    r"짜친다",
    r"잼민이",
    r"새끼들",
    r"ㅇㄱㄹ",
    r"아가리",
    r"i got it",
    r"새기",
    r"ㅅ새",
    r"개새끼",

    # 정치 키워드
    r"이재명",
    r"윤석열",
    r"석열이",
    r"계엄령",
    r"공산당",
    r"박근혜",
    r"문재인",
    r"공산주의", 
    r"전체주의",
    r"쿠데타",
    r"북한",
    r"일제",
    r"김정은"
]

combined_pattern = re.compile("|".join(badword_patterns))

# 필터링
filtered_comments = df_read_final_clean[df_read_final_clean["comment"].str.contains(combined_pattern, na=False)]

C:\Users\user\AppData\Local\Temp/ipykernel_32796/3849677040.py:82: UserWarning: This pattern has match groups. To actually get the groups, use str.extract.
  filtered_comments = df_read_final_clean[df_read_final_clean["comment"].str.contains(combined_pattern, na=False)]


In [251]:
print(f"필터링된 댓글 수: {len(filtered_comments)}")
print("=" * 80)

for idx, row in filtered_comments.iterrows():
    print(f"[인덱스: {idx}]")
    print(f"날짜: {row.get('date', 'N/A')}")
    print(f"카테고리: {row.get('category', 'N/A')}")
    print(f"댓글 내용: {row['comment']}")
    print("-" * 80)

필터링된 댓글 수: 1381
[인덱스: 21]
날짜: 2025-01-01 00:30:46
카테고리: 기타
댓글 내용: 텔포는 다시 봐도 짜친다... 이젠 텔레포트가 아니라 스노우볼링이잖냐..
--------------------------------------------------------------------------------
[인덱스: 28]
날짜: 2025-01-01 00:30:46
카테고리: 기타
댓글 내용: 또 안바뀌면 고착화되고 새롭지 못하다고 지랄할거면서ㅋㅋ
--------------------------------------------------------------------------------
[인덱스: 116]
날짜: 2025-01-01 00:30:46
카테고리: 포지션 밸런스
댓글 내용: 패치 진짜 병sin같네
한팀만 신발업글
정글오브젝트추가
무한 쌍둥이포탑젠
포블,첫용,퍼블 강제싸움
바텀지옥+패작+일부러시간끄는 적군+신발업글부터차이나는 기본적인 성장차이+지는팀은 ㅈㄴ하기싫어짐+아타칸 시스템자체가 ㅈ박음+유충챙겨야됨+지는팀 바론,장로기회없음+노잼톤,또바나같은 개노잼 특정챔고정시대 돌입

이번시즌은 버려야겠네
져도 이겨도 압도적으로 지고 압도적으로 이겨서 게임할맛나겠나...
--------------------------------------------------------------------------------
[인덱스: 191]
날짜: 2025-01-01 00:30:46
카테고리: 포지션 밸런스
댓글 내용: 본인 마스터 400점인데 요즘 솔랭특
조합? 그딴거없음 한타생각해서 조합픽해도 초중반 운영에서 개말림
왜냐하면 어느팀 정글이 카정, 유충으로 깽판치냐 싸움이기 때문임. 미드에서 초반 주도권약한 스탠딩메이지나 라인푸쉬느린 픽하면 정글도 교전을 피해야되는데 그냥 습관대로 상대정글과의 운명의 주도권 대결을함 절대 그 조합에맞는 플레이를 안함. 
이새기들도 이 공식에 절여져서 주도권 없는 미드가 싸움피하고싶어해도 상황에 맞게 안하

In [252]:
df_read_final_clean_2= df_read_final_clean.copy()

In [253]:
df_read_final_clean["sensitive_flag"] = df_read_final_clean["comment"].str.contains(combined_pattern,na=False)

C:\Users\user\AppData\Local\Temp/ipykernel_32796/2088479124.py:1: UserWarning: This pattern has match groups. To actually get the groups, use str.extract.
  df_read_final_clean["sensitive_flag"] = df_read_final_clean["comment"].str.contains(combined_pattern,na=False)


In [222]:
df_read_final_clean_2

,date,source,region_type,comment,main_position,category,complaint_index,sensitive_flag
0,2025-01-01 00:30:46,youtube,국내,8:46 신분제 도입,기타,기타,0,False
1,2025-01-01 00:30:46,youtube,국내,명예 5렙상점 보상이나 추가해줬으면... 명예 5렙토큰만 4개 쌓여 있는데 트위치랑...,기타,챔피언 언급,-1,False
2,2025-01-01 00:30:46,youtube,국내,다들 새해 복 많이 받아라,기타,기타,0,False
3,2025-01-01 00:30:46,youtube,국내,근데 신발은 한팀만 강화시키는건 진짜 안좋은 판단같음... 지금처럼 무료로 업그레이...,기타,전투/데미지,1,False
4,2025-01-01 00:30:46,youtube,국내,나이스 드디어 명예레벨 낮은 것들이 채팅을 못치는 시대가 오는고만,기타,게임 시스템,-1,False
...,...,...,...,...,...,...,...,...
152911,2025-10-26 07:18:25,reddit,해외,"우선권이 있다면, 정글러가 주변에 없을 때 언제든 공격을 시작할 수 있고, 그 후 ...",정글,포지션 밸런스,-1,False
152912,2025-12-02 10:51:24,reddit,해외,"예를 들어, 아트록스는 레벨당 대략 5의 공격력을 얻습니다. 레벨이 2만큼 더 오르...",기타,챔피언 언급,-1,False
152913,2025-12-02 14:45:32,reddit,해외,스테락스가 약간 더 나을 겁니다. 하지만 보너스 6 광고가 있다고 해서 대부분의 아...,기타,전투/데미지,0,False
152914,2025-12-05 16:21:43,reddit,해외,"솔직히 너무 과하게 생각하시는 것 같아요. 전 인내력이 필요할 때만 연승을 가고, ...",기타,기타,0,False


In [254]:
df_read_final_clean

,date,source,region_type,comment,main_position,category,complaint_index,sensitive_flag
0,2025-01-01 00:30:46,youtube,국내,8:46 신분제 도입,기타,기타,0,False
1,2025-01-01 00:30:46,youtube,국내,명예 5렙상점 보상이나 추가해줬으면... 명예 5렙토큰만 4개 쌓여 있는데 트위치랑...,기타,챔피언 언급,-1,False
2,2025-01-01 00:30:46,youtube,국내,다들 새해 복 많이 받아라,기타,기타,0,False
3,2025-01-01 00:30:46,youtube,국내,근데 신발은 한팀만 강화시키는건 진짜 안좋은 판단같음... 지금처럼 무료로 업그레이...,기타,전투/데미지,1,False
4,2025-01-01 00:30:46,youtube,국내,나이스 드디어 명예레벨 낮은 것들이 채팅을 못치는 시대가 오는고만,기타,게임 시스템,-1,False
...,...,...,...,...,...,...,...,...
152911,2025-10-26 07:18:25,reddit,해외,"우선권이 있다면, 정글러가 주변에 없을 때 언제든 공격을 시작할 수 있고, 그 후 ...",정글,포지션 밸런스,-1,False
152912,2025-12-02 10:51:24,reddit,해외,"예를 들어, 아트록스는 레벨당 대략 5의 공격력을 얻습니다. 레벨이 2만큼 더 오르...",기타,챔피언 언급,-1,False
152913,2025-12-02 14:45:32,reddit,해외,스테락스가 약간 더 나을 겁니다. 하지만 보너스 6 광고가 있다고 해서 대부분의 아...,기타,전투/데미지,0,False
152914,2025-12-05 16:21:43,reddit,해외,"솔직히 너무 과하게 생각하시는 것 같아요. 전 인내력이 필요할 때만 연승을 가고, ...",기타,기타,0,False


In [255]:
df_read_final_clean["category"] = df_read_final_clean["comment"].apply(classify_comment)

In [259]:
df_read_final_clean

,date,source,region_type,comment,main_position,category,complaint_index,sensitive_flag
0,2025-01-01 00:30:46,youtube,국내,8:46 신분제 도입,기타,기타,0,False
1,2025-01-01 00:30:46,youtube,국내,명예 5렙상점 보상이나 추가해줬으면... 명예 5렙토큰만 4개 쌓여 있는데 트위치랑...,기타,챔피언 언급,-1,False
2,2025-01-01 00:30:46,youtube,국내,다들 새해 복 많이 받아라,기타,기타,0,False
3,2025-01-01 00:30:46,youtube,국내,근데 신발은 한팀만 강화시키는건 진짜 안좋은 판단같음... 지금처럼 무료로 업그레이...,기타,전투/데미지,1,False
4,2025-01-01 00:30:46,youtube,국내,나이스 드디어 명예레벨 낮은 것들이 채팅을 못치는 시대가 오는고만,기타,게임 시스템,-1,False
...,...,...,...,...,...,...,...,...
152911,2025-10-26 07:18:25,reddit,해외,"우선권이 있다면, 정글러가 주변에 없을 때 언제든 공격을 시작할 수 있고, 그 후 ...",정글,포지션 밸런스,-1,False
152912,2025-12-02 10:51:24,reddit,해외,"예를 들어, 아트록스는 레벨당 대략 5의 공격력을 얻습니다. 레벨이 2만큼 더 오르...",기타,챔피언 언급,-1,False
152913,2025-12-02 14:45:32,reddit,해외,스테락스가 약간 더 나을 겁니다. 하지만 보너스 6 광고가 있다고 해서 대부분의 아...,기타,전투/데미지,0,False
152914,2025-12-05 16:21:43,reddit,해외,"솔직히 너무 과하게 생각하시는 것 같아요. 전 인내력이 필요할 때만 연승을 가고, ...",기타,기타,0,False


In [262]:
all_comments = " ".join(df_read_final_clean["comment"].astype(str))

not_mentioned = []

for champ in champion_list:
    if not re.search(re.escape(champ), all_comments):
        not_mentioned.append(champ)

print("댓글에서 한 번도 언급되지 않은 챔피언:")
print(not_mentioned)
print("개수:", len(not_mentioned))

댓글에서 한 번도 언급되지 않은 챔피언:
['문도박사미스 포츈', '미스포춘', '자르반4세']
개수: 3


In [258]:
df_read_final_clean.to_csv(
    r"F:\porti\2025\스파르타자료들\python_study\venv\04_Team8\0.googledrive\comment_youtube_reddit_cleaned_data.csv",
    index=False,
    encoding="utf-8-sig"
)